# KaggleResearch v3.0

**Autonomous ML research for Kaggle competitions**

This notebook provides a two-phase workflow:

1. **Setup Research** - Download data, analyze structure, generate baseline, run baseline
2. **Run Research** - Unified autonomous loop: literature review → strategy → ideate → test → review → (re-research if needed) → repeat

---

## Cell 1: Install Dependencies & Configuration

**Edit the configuration values before running!**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

!pip install -q kaggle anthropic arxiv segmentation_models_pytorch \
    optuna catboost polars sentence-transformers tavily-python

# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

# Clone KaggleResearch from GitHub
import os
import sys

KAGGLERESEARCH_PATH = '/content/kaggleresearch'
BRANCH = 'main'  # Change to use a specific branch (e.g., 'refactor-research-loop-add-tests')

if os.path.exists(KAGGLERESEARCH_PATH):
    print(f"Updating kaggleresearch (branch: {BRANCH})...")
    !cd {KAGGLERESEARCH_PATH} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"Cloning kaggleresearch (branch: {BRANCH})...")
    !git clone -b {BRANCH} https://github.com/benjaminwfriedman/kaggleresearch.git {KAGGLERESEARCH_PATH}

if KAGGLERESEARCH_PATH not in sys.path:
    sys.path.insert(0, KAGGLERESEARCH_PATH)

print(f"✓ KaggleResearch installed at {KAGGLERESEARCH_PATH}")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION - EDIT THESE VALUES
# ═══════════════════════════════════════════════════════════════════════════════

# API Credentials (required)
KAGGLE_API_TOKEN     = ''       # Your Kaggle API token (kaggle.com/settings > API)
ANTHROPIC_API_KEY    = ''       # Your Anthropic API key
TAVILY_API_KEY       = ''       # (Optional) Tavily API key for web search

# Competition
COMPETITION_URL      = 'https://www.kaggle.com/competitions/titanic'
DRIVE_PATH           = '/content/drive/MyDrive/kaggleresearch'

# Timing & Exploration
TIME_BUDGET_MIN      = 4        # Minutes per experiment (3-5 for Colab)
EXPLORATION_MODE     = 'df'     # 'df' (depth-first) or 'bf' (breadth-first)

# Plateau Detection
PLATEAU_WINDOW       = 5        # Experiments to evaluate for plateau
PLATEAU_MIN_GAIN_PCT = 1.0      # Min % improvement to avoid plateau trigger

# Literature Search
LITERATURE_DEPTH     = 10       # Papers to retrieve (5-20)

# Re-research
MAX_RERESEARCH_ATTEMPTS = 2     # Max re-research attempts before halting (2-5)

# Reset Control
RESET_RUN            = False    # Set True to archive current run and start fresh

# ═══════════════════════════════════════════════════════════════════════════════
# SETUP CREDENTIALS
# ═══════════════════════════════════════════════════════════════════════════════

# Kaggle credentials
if KAGGLE_API_TOKEN:
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod('/root/.kaggle/access_token', 0o600)

# Tavily credentials
if TAVILY_API_KEY:
    os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

# Validate
assert COMPETITION_URL, "Please set COMPETITION_URL"
assert KAGGLE_API_TOKEN, "Please set KAGGLE_API_TOKEN"
assert ANTHROPIC_API_KEY, "Please set ANTHROPIC_API_KEY"

print(f"\nCompetition: {COMPETITION_URL}")
print(f"Drive path: {DRIVE_PATH}")
print(f"Exploration mode: {EXPLORATION_MODE}")
if RESET_RUN:
    print("⚠ RESET_RUN is True - will archive current run")
print("\n✓ Configuration complete!")

## Cell 2: Setup Research

Download competition data, generate baseline model, run baseline.

In [ ]:
# Force reload utils modules (needed after git pull updates)
import importlib
import sys
modules_to_reload = [m for m in sys.modules if m.startswith('utils')]
for m in modules_to_reload:
    del sys.modules[m]

import json
from pathlib import Path
import anthropic

from utils.kaggle_api import extract_slug_from_url, parse_competition, download_competition_data
from utils.checkpoint import (
    load_checkpoint, save_checkpoint, detect_phase,
    create_initial_checkpoint, archive_and_reset, get_or_create_run_id
)
from utils.branching import init_repo
from utils.experiment_runner import generate_baseline, run_experiment, detect_gpu, scale_time_budget

# ═══════════════════════════════════════════════════════════════════════════════
# SETUP PATHS
# ═══════════════════════════════════════════════════════════════════════════════

COMPETITION_SLUG = extract_slug_from_url(COMPETITION_URL)
PROJECT_DIR = Path(DRIVE_PATH) / COMPETITION_SLUG
REPO_DIR = PROJECT_DIR / 'repo'
DATA_DIR = REPO_DIR / 'data'
LITERATURE_DIR = PROJECT_DIR / 'literature_cache'

CHECKPOINT_PATH = PROJECT_DIR / 'checkpoint.json'
STRATEGY_PATH = PROJECT_DIR / 'STRATEGY.md'
IDEAS_PATH = PROJECT_DIR / 'IDEAS.md'
DB_PATH = PROJECT_DIR / 'experiment_log.db'
TREE_PATH = PROJECT_DIR / 'idea_tree.json'
BASELINE_TRAIN_PATH = PROJECT_DIR / 'baseline_train.py'

# Create directories
for d in [PROJECT_DIR, REPO_DIR, DATA_DIR, LITERATURE_DIR, REPO_DIR / 'submissions']:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")

# Handle reset if requested
if RESET_RUN:
    print("\n⚠ Archiving current run...")
    RUN_ID = archive_and_reset(PROJECT_DIR, COMPETITION_SLUG)
else:
    existing_checkpoint = load_checkpoint(CHECKPOINT_PATH)
    RUN_ID = get_or_create_run_id(PROJECT_DIR, existing_checkpoint)

print(f"Run ID: {RUN_ID}")

# Initialize Anthropic client
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Detect GPU and scale time budget
GPU_NAME, HAS_GPU = detect_gpu()
SCALED_TIME_BUDGET = scale_time_budget(TIME_BUDGET_MIN, GPU_NAME)
print(f"GPU: {GPU_NAME}")
print(f"Time budget: {SCALED_TIME_BUDGET:.1f} min/experiment")

# ═══════════════════════════════════════════════════════════════════════════════
# SETUP RESEARCH (runs if in bootstrap phase)
# ═══════════════════════════════════════════════════════════════════════════════

checkpoint = load_checkpoint(CHECKPOINT_PATH)
current_phase = detect_phase(checkpoint)
print(f"\nCurrent phase: {current_phase}")

if current_phase == 'bootstrap' or checkpoint is None:
    print("\n" + "="*50)
    print("=== Setup Research ===")
    print("="*50)
    
    # Parse competition
    print("\nParsing competition metadata...")
    competition_meta = parse_competition(COMPETITION_URL)
    print(f"  Name: {competition_meta.name}")
    print(f"  Metric: {competition_meta.metric}")
    print(f"  Metric direction: {competition_meta.metric_direction}")
    
    # Download data
    print("\nDownloading competition data...")
    downloaded_files = download_competition_data(COMPETITION_SLUG, DATA_DIR)
    print(f"  Downloaded {len(downloaded_files)} files")
    
    # Generate baseline using agent (also infers metric direction if unknown)
    print("\nGenerating baseline train.py...")
    train_py_content, problem_type, inferred_metric_direction = generate_baseline(
        data_dir=DATA_DIR,
        competition_name=competition_meta.name,
        metric=competition_meta.metric,
        metric_direction=competition_meta.metric_direction,
        target_column=competition_meta.target_column,
        id_column=competition_meta.id_column,
        client=client
    )
    
    # Use LLM-inferred metric direction (may differ from API metadata)
    final_metric_direction = inferred_metric_direction
    if final_metric_direction != competition_meta.metric_direction:
        print(f"  Metric direction updated: {competition_meta.metric_direction} -> {final_metric_direction}")
    
    # Write train.py
    train_path = REPO_DIR / 'train.py'
    with open(train_path, 'w') as f:
        f.write(train_py_content)
    print(f"  Problem type: {problem_type}")
    print(f"✓ Generated baseline train.py")
    
    # Cache baseline for pivot reuse
    with open(BASELINE_TRAIN_PATH, 'w') as f:
        f.write(train_py_content)
    
    # Initialize git repo
    init_repo(REPO_DIR)
    print("✓ Initialized git repository")
    
    # Run baseline
    print("\nRunning baseline experiment...")
    baseline_score, error = run_experiment(REPO_DIR, SCALED_TIME_BUDGET)
    
    if baseline_score is None:
        print(f"\n⚠ Baseline failed: {error}")
        print("Fix train.py and re-run this cell")
        baseline_score = 0.0
    else:
        print(f"\n✓ Baseline score: {baseline_score:.6f}")
    
    # Save baseline score
    with open(REPO_DIR / 'baseline_score.txt', 'w') as f:
        f.write(str(baseline_score))
    
    # Create checkpoint with LLM-inferred metric direction
    checkpoint = create_initial_checkpoint(
        competition_slug=COMPETITION_SLUG,
        problem_type=problem_type,
        baseline_score=baseline_score,
        run_id=RUN_ID,
        exploration_mode=EXPLORATION_MODE,
        competition_meta={
            'name': competition_meta.name,
            'metric': competition_meta.metric,
            'metric_direction': final_metric_direction,
        }
    )
    checkpoint.strategy_name = "Baseline"
    save_checkpoint(CHECKPOINT_PATH, checkpoint)
    
    print("\n" + "="*50)
    print("✓ Setup Research complete!")
    print(f"  Run ID: {RUN_ID}")
    print(f"  Problem type: {problem_type}")
    print(f"  Metric direction: {final_metric_direction}")
    print(f"  Baseline score: {baseline_score:.6f}")
    print("="*50)
    print("\nRun Cell 3 to start the research loop.")

else:
    print(f"\nResuming from checkpoint:")
    print(f"  Run ID: {checkpoint.run_id}")
    print(f"  Best score: {checkpoint.best_score:.6f}")
    print(f"  Experiments: {checkpoint.total_experiments}")
    print(f"  Exploration mode: {checkpoint.exploration_mode}")
    print("\nRun Cell 3 to continue research.")

## Cell 3: Run Research

Autonomous research loop: literature review → strategy → ideate → test → review → repeat.

This will run until:
- Session timeout (Colab ~12h limit)
- All research avenues exhausted (halted)
- User interrupts (Ctrl+C)

In [ ]:
# Force reload utils modules (needed after git pull)
import importlib
import sys
modules_to_reload = [m for m in sys.modules if m.startswith('utils')]
for m in modules_to_reload:
    del sys.modules[m]

from IPython.display import clear_output
from pathlib import Path

from utils.research_loop import ResearchConfig, run_research
from utils.experiment_runner import load_experiments
from utils.display import render_live_table, display_in_colab, render_summary
from utils.checkpoint import load_checkpoint
from utils.idea_tree import IdeaTree

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD CONFIG
# ═══════════════════════════════════════════════════════════════════════════════

config = ResearchConfig(
    project_dir=PROJECT_DIR,
    repo_dir=REPO_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    strategy_path=STRATEGY_PATH,
    ideas_path=IDEAS_PATH,
    db_path=DB_PATH,
    tree_path=TREE_PATH,
    literature_dir=LITERATURE_DIR,
    kaggleresearch_path=Path(KAGGLERESEARCH_PATH),
    time_budget_min=SCALED_TIME_BUDGET,
    plateau_window=PLATEAU_WINDOW,
    plateau_min_gain_pct=PLATEAU_MIN_GAIN_PCT,
    exploration_mode=EXPLORATION_MODE,
    literature_depth=LITERATURE_DEPTH,
    max_reresearch_attempts=MAX_RERESEARCH_ATTEMPTS,
)

# Get metric direction from checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)
comp_meta = checkpoint.competition_meta or {}
config.metric_direction = comp_meta.get('metric_direction', 'higher_better')

# ═══════════════════════════════════════════════════════════════════════════════
# DISPLAY CALLBACK
# ═══════════════════════════════════════════════════════════════════════════════

def on_experiment_complete(experiments, checkpoint, tree):
    """Update display after each experiment."""
    clear_output(wait=True)
    html = render_live_table(
        experiments,
        checkpoint.current_branch,
        checkpoint.best_score,
        checkpoint.baseline_score,
        config.metric_direction
    )
    display_in_colab(html)
    print(f"\n{tree.render_tree(config.metric_direction)}")

# ═══════════════════════════════════════════════════════════════════════════════
# RUN RESEARCH
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting autonomous research loop...")
print("(Interrupt with Ctrl+C to stop early)\n")

result = run_research(config, client, display_callback=on_experiment_complete)

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("=== Research Complete ===")
print("="*60)
print(f"\nFinal score: {result.final_score:.6f}")
print(f"Baseline: {result.baseline_score:.6f}")
print(f"Improvement: {result.final_score - result.baseline_score:+.6f}")
print(f"Experiments: {result.total_experiments} ({result.successful_experiments} improved)")
print(f"Exit reason: {result.exit_reason}")

# Generate and display summary
checkpoint = load_checkpoint(CHECKPOINT_PATH)
summary_md = render_summary(
    log_path=DB_PATH,
    checkpoint=checkpoint.to_dict(),
    baseline_score=checkpoint.baseline_score,
    strategy_md_path=STRATEGY_PATH,
    tree_path=TREE_PATH
)

# Save summary
with open(PROJECT_DIR / 'SUMMARY.md', 'w') as f:
    f.write(summary_md)
print(f"\n✓ Summary saved to {PROJECT_DIR / 'SUMMARY.md'}")

# Show exploration tree
print("\n=== Final Exploration Tree ===")
tree = IdeaTree(TREE_PATH)
if tree.load():
    print(tree.render_tree(config.metric_direction))

# Show submission location
submission_path = REPO_DIR / 'submissions' / 'submission.csv'
if submission_path.exists():
    print(f"\nSubmission file: {submission_path}")
    print("Submit this file to Kaggle to see your leaderboard score.")